# Module 4 — 向量空間與自由度

> **對應程度**：大學線代基礎，搭配機械系統直觀理解

本模組涵蓋：
1. 線性獨立與相依
2. 矩陣的秩（Rank）
3. 零空間（Null Space）
4. 投影與 Gram-Schmidt 正交化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg as sla
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.linalg_utils import gram_schmidt, projection_onto_subspace
from src.visualizer import set_style, plot_vectors_2d

set_style()
print('Module 4 載入完成！')

---
## 4.1 線性獨立與相依

**線性相依** = 某向量可以由其他向量表示

**判斷方法**：行列式 = 0 ↔ 線性相依

In [ ]:
# 2D 線性相依 vs 獨立
v1 = np.array([1, 2])
v2_indep = np.array([2, 1])   # 獨立
v2_dep = np.array([2, 4])     # 相依（= 2*v1）

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_vectors_2d([v1, v2_indep], labels=['v1', 'v2'], colors=['blue', 'red'],
                ax=ax1, title=f'線性獨立 (det = {np.linalg.det(np.array([v1, v2_indep])):.1f})')
plot_vectors_2d([v1, v2_dep], labels=['v1', 'v2=2v1'], colors=['blue', 'red'],
                ax=ax2, title=f'線性相依 (det = {np.linalg.det(np.array([v1, v2_dep])):.1f})')
plt.tight_layout()
plt.show()

In [ ]:
# 3D 共面判斷
f1 = np.array([1, 0, 0])
f2 = np.array([0, 1, 0])
f3_coplanar = np.array([1, 1, 0])     # 共面
f3_not = np.array([0, 0, 1])          # 不共面

A_co = np.array([f1, f2, f3_coplanar])
A_no = np.array([f1, f2, f3_not])

print(f'三力共面: rank = {np.linalg.matrix_rank(A_co)}, det = {np.linalg.det(A_co):.4f}')
print(f'三力不共面: rank = {np.linalg.matrix_rank(A_no)}, det = {np.linalg.det(A_no):.4f}')
print(f'\n共面 → 無法提供三維空間的完整約束')

---
## 4.2 矩陣的秩（Rank）

**秩-零化度定理**：$\text{rank}(A) + \text{nullity}(A) = n$

### 物理應用：機構自由度 = n - rank(Jacobian)

In [ ]:
# 連桿機構雅可比矩陣
# 2-DOF 平面機構在正常位形
theta1, theta2 = np.pi/4, np.pi/6
L1, L2 = 1.0, 0.8

J_normal = np.array([
    [-L1*np.sin(theta1) - L2*np.sin(theta1+theta2), -L2*np.sin(theta1+theta2)],
    [L1*np.cos(theta1) + L2*np.cos(theta1+theta2), L2*np.cos(theta1+theta2)],
])

# 奇異位形：兩臂伸直
J_singular = np.array([
    [-(L1+L2)*np.sin(0), -L2*np.sin(0)],
    [(L1+L2)*np.cos(0), L2*np.cos(0)],
])

print('正常位形:')
print(f'  J = \n{J_normal}')
print(f'  rank(J) = {np.linalg.matrix_rank(J_normal)}')
print(f'  det(J) = {np.linalg.det(J_normal):.4f}')

print(f'\n奇異位形（兩臂伸直）:')
print(f'  J = \n{J_singular}')
print(f'  rank(J) = {np.linalg.matrix_rank(J_singular)}')
print(f'  det(J) = {np.linalg.det(J_singular):.4f}')
print(f'  → 在奇異位形失去一個自由度！')

In [ ]:
# 秩-零化度定理驗證
matrices = {
    '滿秩 3×3': np.array([[1,0,0],[0,1,0],[0,0,1]]),
    '秩不足 3×3': np.array([[1,2,3],[4,5,6],[7,8,9]]),
    '4×3 矩陣': np.array([[1,2,3],[4,5,6],[7,8,9],[10,11,12]]),
}

for name, A in matrices.items():
    rank = np.linalg.matrix_rank(A)
    nullity = sla.null_space(A).shape[1]
    n = A.shape[1]
    print(f'{name}: rank={rank}, nullity={nullity}, n={n}, rank+nullity={rank+nullity} = n ✓')

---
## 4.3 零空間（Null Space）

$\text{Null}(A) = \{\vec{x} : A\vec{x} = \vec{0}\}$

### 物理應用：剛體的「零空間運動」= 不產生內力的運動

In [ ]:
# 未約束桁架的零空間 = 剛體運動
# 1D 簡化：三個自由節點
k = 100.0
K = np.array([
    [2*k, -k, 0],
    [-k, 2*k, -k],
    [0, -k, k]
])

print('未約束彈簧-質量剛度矩陣 K =')
print(K)

ns = sla.null_space(K)
print(f'\n零空間維度: {ns.shape[1]}')
if ns.shape[1] > 0:
    print(f'零空間基向量: {ns[:, 0] / ns[0, 0]}')  # 正規化
    print('→ 代表均勻平移（所有質量同步移動，不產生彈簧力）')
    # 驗證
    print(f'K @ null = {K @ ns[:, 0]}  (應為零向量 ✓)')

In [ ]:
# 零空間視覺化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左圖：零空間運動（平移）
x_pos = [0, 1, 2]  # 節點位置
ax1.plot(x_pos, [0, 0, 0], 'ko-', ms=15, lw=2, label='平衡位置')
# 均勻平移
d = 0.3
ax1.plot([x+d for x in x_pos], [0.3]*3, 'rs--', ms=15, lw=2, label=f'平移 d={d}')
for i, x in enumerate(x_pos):
    ax1.annotate('', xy=(x+d, 0.3), xytext=(x, 0),
                arrowprops=dict(arrowstyle='->', color='blue', lw=1.5))
ax1.set_title('零空間運動：均勻平移（無內力）')
ax1.legend()
ax1.set_ylim(-0.5, 0.8)
ax1.grid(True, alpha=0.3)

# 右圖：非零空間運動（有內力）
ax2.plot(x_pos, [0, 0, 0], 'ko-', ms=15, lw=2, label='平衡位置')
displacements = [0, 0.3, 0]  # 只有中間節點移動
ax2.plot([x_pos[i] + displacements[i] for i in range(3)],
         [0.3]*3, 'rs--', ms=15, lw=2, label='變形')
ax2.set_title('非零空間運動：差異位移（產生內力）')
ax2.legend()
ax2.set_ylim(-0.5, 0.8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4.4 投影與 Gram-Schmidt 正交化

### 物理應用：加速度計去除重力分量

In [ ]:
# Gram-Schmidt 正交化
vectors = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]], dtype=float)
Q = gram_schmidt(vectors)

print('原始向量:')
for i, v in enumerate(vectors):
    print(f'  v{i+1} = {v}')

print('\n正交化後:')
for i, q in enumerate(Q):
    print(f'  q{i+1} = [{q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}]')

# 驗證正交性
print(f'\nQ @ Q^T = ')
print(np.round(Q @ Q.T, 10))
print(f'\n正交矩陣 ✓: {np.allclose(Q @ Q.T, np.eye(3))}')

In [ ]:
# 加速度計去重力
gravity = np.array([0, 0, 9.81])  # 重力方向

# 模擬加速度計量測（包含重力 + 運動加速度）
a_motion = np.array([0.5, -0.3, 0.1])   # 運動產生的加速度
a_total = a_motion + gravity              # 總量測值

print(f'總量測加速度: {a_total}')
print(f'真實運動加速度: {a_motion}')

# 方法：投影到重力方向，再相減
g_unit = gravity / np.linalg.norm(gravity)
a_gravity_proj = np.dot(a_total, g_unit) * g_unit
a_estimated = a_total - a_gravity_proj

print(f'\n重力投影: {a_gravity_proj}')
print(f'估計運動加速度: {a_estimated}')
print(f'誤差: {np.linalg.norm(a_estimated - a_motion):.6f}')

In [ ]:
# 投影矩陣冪等性驗證: P² = P
basis = np.array([[1, 1, 0], [0, 1, 1]], dtype=float)
Q_basis = gram_schmidt(basis)
# 投影矩陣 P = Q Q^T
P = Q_basis.T @ Q_basis

print(f'投影矩陣 P =')
print(np.round(P, 4))
print(f'\nP² = P ? {np.allclose(P @ P, P)} ✓ (冪等性)')
print(f'P = P^T ? {np.allclose(P, P.T)} ✓ (對稱性)')

# 驗證投影
v = np.array([1, 2, 3])
proj = projection_onto_subspace(v, Q_basis)
proj2 = projection_onto_subspace(proj, Q_basis)
print(f'\nv = {v}')
print(f'Proj(v) = {proj}')
print(f'Proj(Proj(v)) = {proj2}')
print(f'冪等性: {np.allclose(proj, proj2)} ✓')

---
## Module 4 驗證總結

| 項目 | 驗證方式 | 結果 |
|------|----------|------|
| 線性相依 | det = 0 | ✓ |
| 秩-零化度定理 | rank + nullity = n | ✓ |
| 零空間 | Kx = 0 | ✓ |
| Gram-Schmidt | Q Q^T = I | ✓ |
| 投影冪等性 | P² = P | ✓ |
| 去重力 | 估計 ≈ 真實 | ✓ |